# Remote Coding Agent on Google Colab

Launch the authenticated MCP server through a temporary Cloudflare Quick Tunnel. Run cells in order. Colab is ephemeral, and the tunnel is public, so use short-lived, narrowly scoped credentials only.

Add `MCP_AUTH_TOKEN` to Colab Secrets. Add `GITHUB_TOKEN` only when cloning a private repository or when GitHub access is explicitly needed. Never place secrets in notebook output. Commands are powerful and must be treated as untrusted code.

In [ ]:
from getpass import getpass

from google.colab import userdata


def read_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = getpass(f"Enter {name} (input hidden): ")
    value = value.strip()
    if required and not value:
        raise ValueError(f"{name} is required")
    return value


MCP_AUTH_TOKEN = read_secret("MCP_AUTH_TOKEN")
GITHUB_TOKEN = read_secret("GITHUB_TOKEN", required=False)
if len(MCP_AUTH_TOKEN) < 32:
    raise ValueError("Use a high-entropy MCP_AUTH_TOKEN of at least 32 characters")
print("Secrets loaded; values are not displayed.")

In [ ]:
import os
import pathlib
import subprocess

REPO_DIR = pathlib.Path("/content/remote-coding-agent")
ASKPASS = pathlib.Path("/content/.git-askpass.sh")
clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"

if GITHUB_TOKEN:
    ASKPASS.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '  *Username*) printf "%s" "x-access-token" ;;\n'
        '  *) printf "%s" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    ASKPASS.chmod(0o700)
    clone_env.update(
        {"GITHUB_TOKEN": GITHUB_TOKEN, "GIT_ASKPASS": str(ASKPASS)}
    )

try:
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
            env=clone_env,
            check=True,
            timeout=120,
        )
    elif REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository; inspect it before retrying"
        )
    else:
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/Dishant402955/remote-coding-agent.git",
                str(REPO_DIR),
            ],
            env=clone_env,
            check=True,
            timeout=120,
        )
finally:
    ASKPASS.unlink(missing_ok=True)
    clone_env.pop("GITHUB_TOKEN", None)
print(f"Repository ready at {REPO_DIR}")

In [ ]:
%cd /content/remote-coding-agent
%pip -q install -e . httpx
print("Project dependencies installed.")

In [ ]:
import hashlib
import json
import pathlib
import platform
import subprocess
import urllib.request

asset_names = {
    "x86_64": "cloudflared-linux-amd64.deb",
    "amd64": "cloudflared-linux-amd64.deb",
    "aarch64": "cloudflared-linux-arm64.deb",
    "arm64": "cloudflared-linux-arm64.deb",
}
asset_name = asset_names.get(platform.machine().lower())
if not asset_name:
    raise RuntimeError(f"Unsupported Colab architecture: {platform.machine()}")

release_url = (
    "https://api.github.com/repos/cloudflare/cloudflared/releases/latest"
)
request = urllib.request.Request(
    release_url,
    headers={"Accept": "application/vnd.github+json"},
)
with urllib.request.urlopen(request, timeout=20) as response:
    release = json.load(response)
asset = next(
    (item for item in release["assets"] if item["name"] == asset_name),
    None,
)
if asset is None:
    raise RuntimeError(f"Could not find {asset_name} in the latest release")
expected_digest = asset.get("digest")
if not expected_digest or not expected_digest.startswith("sha256:"):
    raise RuntimeError(
        "Release metadata has no SHA-256 digest; refusing an unverifiable download"
    )

deb_path = pathlib.Path("/content/cloudflared.deb")
hasher = hashlib.sha256()
with urllib.request.urlopen(asset["browser_download_url"], timeout=30) as response:
    with deb_path.open("wb") as destination:
        total_bytes = 0
        while chunk := response.read(1024 * 1024):
            total_bytes += len(chunk)
            if total_bytes > 200 * 1024 * 1024:
                raise RuntimeError("cloudflared package exceeds the 200 MiB limit")
            hasher.update(chunk)
            destination.write(chunk)

actual_digest = f"sha256:{hasher.hexdigest()}"
if actual_digest != expected_digest:
    deb_path.unlink(missing_ok=True)
    raise RuntimeError("cloudflared SHA-256 verification failed")

subprocess.run(
    ["dpkg", "-i", str(deb_path)],
    check=True,
    stdout=subprocess.DEVNULL,
)
subprocess.run(["cloudflared", "--version"], check=True)
print(f"Verified cloudflared package for release {release['tag_name']}")

In [ ]:
import os
import pathlib
import re
import signal
import subprocess
import time

workspace = pathlib.Path("/content/workspace")
workspace.mkdir(parents=True, exist_ok=True)

# Never forward the optional clone token to the server process.
server_env = os.environ.copy()
for secret_name in ("GITHUB_TOKEN", "GH_TOKEN", "MCP_AUTH_TOKEN"):
    server_env.pop(secret_name, None)
server_env.update(
    {
        "MCP_AUTH_TOKEN": MCP_AUTH_TOKEN,
        "REMOTE_CODING_AGENT_HOST": "127.0.0.1",
        "REMOTE_CODING_AGENT_PORT": "8000",
        "REMOTE_CODING_AGENT_WORKSPACE": str(workspace),
    }
)

def stop_process(process):
    if process is None or process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=5)

# Clean up a previous cell run before starting replacement processes.
stop_process(globals().get("tunnel"))
stop_process(globals().get("server"))

server_log = pathlib.Path("/content/remote-coding-agent-server.log")
server_log_handle = server_log.open("w", encoding="utf-8")
server = subprocess.Popen(
    ["remote-coding-agent"],
    cwd=REPO_DIR,
    env=server_env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
deadline = time.monotonic() + 15
while time.monotonic() < deadline:
    if server.poll() is not None:
        server_log_handle.close()
        raise RuntimeError(
            f"Server exited during startup. Inspect {server_log} (do not share logs "
            "if they contain sensitive project output)."
        )
    try:
        import httpx

        probe = httpx.get("http://127.0.0.1:8000/", timeout=1)
        if probe.status_code in (200, 401, 404, 405):
            break
    except Exception:
        time.sleep(0.25)
else:
    stop_process(server)
    server_log_handle.close()
    raise RuntimeError("Server did not become reachable within 15 seconds")

tunnel_log = pathlib.Path("/content/cloudflared.log")
tunnel_log_handle = tunnel_log.open("w", encoding="utf-8")
tunnel = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8000",
    ],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

tunnel_url = None
deadline = time.monotonic() + 45
while time.monotonic() < deadline:
    if tunnel.poll() is not None:
        break
    try:
        log_text = tunnel_log.read_text(encoding="utf-8", errors="replace")
    except OSError:
        log_text = ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.25)

tunnel_log_handle.close()
server_log_handle.close()
if not tunnel_url:
    stop_process(tunnel)
    stop_process(server)
    raise RuntimeError(
        f"Tunnel URL was not detected. Inspect {tunnel_log} and rerun after "
        "resolving the cause."
    )

print("MCP server and temporary HTTPS tunnel started.")
print("Endpoint:", tunnel_url + "/mcp")
print("Keep this runtime active. Stop the processes before closing the session.")

In [ ]:
import json

import httpx

LOCAL_ENDPOINT = "http://127.0.0.1:8000/mcp"
HEADERS = {
    "Authorization": f"Bearer {MCP_AUTH_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
}


def mcp_payload(response):
    if "application/json" in response.headers.get("content-type", ""):
        return response.json()
    data_lines = [
        line.removeprefix("data:").strip()
        for line in response.text.splitlines()
        if line.startswith("data:")
    ]
    if not data_lines:
        raise RuntimeError(f"MCP response had no data event: HTTP {response.status_code}")
    return json.loads(data_lines[-1])


with httpx.Client(timeout=10) as client:
    unauthorized = client.post(
        LOCAL_ENDPOINT,
        headers={"Content-Type": "application/json"},
        json={"jsonrpc": "2.0", "id": 0, "method": "initialize", "params": {}},
    )
    if unauthorized.status_code != 401:
        raise RuntimeError("Unauthenticated request was not rejected")

    initialized = client.post(
        LOCAL_ENDPOINT,
        headers=HEADERS,
        json={
            "jsonrpc": "2.0",
            "id": 1,
            "method": "initialize",
            "params": {
                "protocolVersion": "2025-03-26",
                "capabilities": {},
                "clientInfo": {"name": "colab-smoke-check", "version": "1.0"},
            },
        },
    )
    if initialized.status_code != 200:
        raise RuntimeError(
            f"Authenticated MCP initialize failed: HTTP {initialized.status_code}"
        )
    session_id = initialized.headers.get("mcp-session-id")
    if not session_id:
        raise RuntimeError("MCP initialize response did not include a session ID")
    protocol_version = mcp_payload(initialized)["result"]["protocolVersion"]

    session_headers = {
        **HEADERS,
        "MCP-Session-Id": session_id,
        "MCP-Protocol-Version": protocol_version,
    }
    client.post(
        LOCAL_ENDPOINT,
        headers=session_headers,
        json={"jsonrpc": "2.0", "method": "notifications/initialized"},
    )
    tool_response = client.post(
        LOCAL_ENDPOINT,
        headers=session_headers,
        json={
            "jsonrpc": "2.0",
            "id": 2,
            "method": "tools/call",
            "params": {
                "name": "run_command",
                "arguments": {"command": "printf colab-smoke-ok"},
            },
        },
    )
    if tool_response.status_code != 200:
        raise RuntimeError(
            f"run_command failed: HTTP {tool_response.status_code}"
        )

    result = mcp_payload(tool_response)["result"]
    structured = result.get("structuredContent")
    if structured is None:
        structured = json.loads(result["content"][0]["text"])
    if structured.get("stdout") != "colab-smoke-ok":
        raise RuntimeError(f"Unexpected tool result: {structured}")
    if structured.get("exit_code") != 0:
        raise RuntimeError(f"Harmless command failed: {structured}")

print("Authentication, MCP initialization, and run_command smoke checks passed.")
print("This verifies the local service, not ChatGPT/client compatibility.")

## Connect your MCP client

Use the printed HTTPS endpoint ending in `/mcp` and configure `Authorization: Bearer <MCP_AUTH_TOKEN>`. Never put the token in the URL. The tunnel URL changes after runtime restart; server processes and workspace files may disappear when Colab resets.

### Stop the service

Run this cell before ending the runtime:

```python
stop_process(globals().get("tunnel"))
stop_process(globals().get("server"))
print("Server and tunnel stopped.")
```

The quick tunnel is public and temporary. Authentication is required, but it is not a substitute for restricting what commands can access. Do not use valuable credentials or sensitive workspaces in a disposable Colab environment.